# Crea datasets per AIRL con `next_observation`

Genera (o rigenera) i due file usati da `AirlAlgorithm`:

| File | Contenuto |
|---|---|
| `data_for_training/expert_trajectories_airl.pkl` | Lista di `Trajectory` raccolte dal modello PPO expert |
| `data_for_training/balanced_eval_dataset_airl.pkl` | Dataset bilanciato per validazione del reward model |

Entrambi i file vengono generati con `EnvBufferingWrapper`, che setta sempre `next_observation` su ogni `Transition`. Questo è necessario per `AirlRewardNet.shaped_reward()`, che usa `V(s')` sul next state.

**Fonti per il balanced eval dataset:**
- `running` / `arrived` → expert puro (`ppo-fast/model.zip`)
- `collision` / `offroad` → expert con esplorazione casuale (`eps=0.85`)
- `timeout` → `ppo_chri_binary_bernoulli q=50000 seed=2` (unico agente che produce timeout)

In [12]:
from pathlib import Path
from collections import Counter
import pickle
import time

import numpy as np
import sys

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for rel in ["human-feedback-rl", "sumo-rl-ego"]:
    p = str(REPO_ROOT / rel)
    if p not in sys.path:
        sys.path.insert(0, p)

import sumo_rl_ego as sre
from stable_baselines3 import PPO

from human_feedback_rl.common.env_wrappers import EnvBufferingWrapper, PolicyExplorationWrapper
from human_feedback_rl.common.trajectory_generators import rollout_agent
from human_feedback_rl.common.types import Trajectory

# ── Paths ──────────────────────────────────────────────────────────────────
EXPERT_MODEL_PATH = (
    REPO_ROOT / "sumo-rl-ego" / "sumo_rl_ego" / "policies" / "models" / "ppo-fast" / "model.zip"
)
# Agente che produce quasi esclusivamente episodi timeout
# (50 timeout su 54 episodi in 30k step — unico nel set sperimentale)
TIMEOUT_AGENT_PATH = (
    REPO_ROOT
    / "outputs/reward_label_experiments/20260610_092001"
    / "ppo_chri_binary_bernoulli seg=1 q=50000 temp=20 seed=2 bernoulli_50k"
    / "checkpoint_0100"
    / "agent.zip"
)

EXPERT_SAVE_PATH = REPO_ROOT / "data_for_training" / "expert_trajectories_airl.pkl"
EVAL_SAVE_PATH   = REPO_ROOT / "data_for_training" / "balanced_eval_dataset_airl.pkl"

# ── Parametri expert trajectories ──────────────────────────────────────────
N_EXPERT_STEPS = 500_000   # ~3000 traiettorie con mean_len~160
N_EXPERT_ENVS  = 4
EXPERT_SEED    = 42

# ── Parametri balanced eval dataset ────────────────────────────────────────
N_PER_CLASS     = 300
SEGMENT_LENGTHS = [1, 5, 20]
TARGET_CLASSES  = ["running", "arrived", "collision", "offroad", "timeout"]

N_EVAL_STEPS_PER_PASS = 50_000   # per le passate expert / esplorativa
N_TIMEOUT_STEPS       = 200_000  # ~330 timeout ep (50 timeout / 30k step → 180k per 300)
N_EVAL_ENVS           = 4
EVAL_SEED             = 0
RNG_SEED              = 0
rng = np.random.default_rng(RNG_SEED)

print("Repo:", REPO_ROOT)
assert EXPERT_MODEL_PATH.exists(),  f"Expert model non trovato: {EXPERT_MODEL_PATH}"
assert TIMEOUT_AGENT_PATH.exists(), f"Timeout agent non trovato: {TIMEOUT_AGENT_PATH}"
print("Tutti i path trovati — OK")

Repo: /Users/andreazhang/Desktop/newTesi/sumo-human-feedback-rl
Tutti i path trovati — OK


## Helper

In [13]:
STATUS_NAMES = {
    0: "arrived", 1: "collision", 2: "offroad",
    3: "timeout",  4: "running",   5: "teleported", 6: "removed",
}


def transition_status_name(t):
    if t.next_status is None:
        return "unknown"
    return STATUS_NAMES.get(int(np.argmax(t.next_status)), "unknown")


def segment_status_name(segment):
    return transition_status_name(segment[-1])


def make_env(seed=0, n_envs=1):
    return sre.make_vec_env(
        "HighwayEgo-v0", n_envs=n_envs, base_seed=seed,
        ego="continuous", reward="fast",
    )


def collect_trajectories(policy, n_steps, n_envs, seed=0, deterministic=True):
    """Raccoglie traiettorie con next_observation sempre settato."""
    env = make_env(seed=seed, n_envs=n_envs)
    buffering = EnvBufferingWrapper(env, error_on_premature_reset=False)
    try:
        rollout_agent(policy, buffering, steps=n_steps, deterministic_policy=deterministic)
        return buffering.pop_finished_trajectories()
    finally:
        try:
            env.close()
        except Exception:
            pass


def collect_trajectories_with_exploration(policy, n_steps, n_envs, exploration_eps, seed=0):
    """Raccoglie traiettorie con epsilon-esplorazione."""
    env = make_env(seed=seed, n_envs=n_envs)
    buffering = EnvBufferingWrapper(env, error_on_premature_reset=False)
    explorer = PolicyExplorationWrapper(
        venv=env, policy=policy, exploration_eps=exploration_eps,
        rng=np.random.default_rng(seed),
    )
    try:
        rollout_agent(explorer, buffering, steps=n_steps, deterministic_policy=False)
        return buffering.pop_finished_trajectories()
    finally:
        try:
            env.close()
        except Exception:
            pass


def verify_next_observation(trajectories, label=""):
    total = sum(len(t) for t in trajectories)
    missing = sum(1 for t in trajectories for tr in t if tr.next_observation is None)
    ok = missing == 0
    tag = "OK" if ok else f"ERRORE: {missing} mancanti!"
    print(f"{label}: {len(trajectories)} traj, {total} trans — next_observation: {tag}")
    return ok


print("Helper definiti.")

Helper definiti.


## 1. Expert Trajectories

In [ ]:
print("Caricamento modello expert...")
env_tmp = make_env(seed=EXPERT_SEED, n_envs=1)
expert_policy = PPO.load(str(EXPERT_MODEL_PATH), env=env_tmp, device="cpu")
env_tmp.close()

print(f"Raccolta {N_EXPERT_STEPS:,} step ({N_EXPERT_ENVS} env paralleli)...")
t0 = time.perf_counter()
expert_trajectories = collect_trajectories(
    policy=expert_policy, n_steps=N_EXPERT_STEPS,
    n_envs=N_EXPERT_ENVS, seed=EXPERT_SEED, deterministic=True,
)
elapsed = time.perf_counter() - t0

lengths = [len(t) for t in expert_trajectories]
print(f"Completato in {elapsed:.1f}s")
print(f"  n_trajectories   : {len(expert_trajectories)}")
print(f"  total_transitions: {sum(lengths):,}")
print(f"  min/mean/max len : {min(lengths)} / {np.mean(lengths):.1f} / {max(lengths)}")
print(f"  terminali        : {dict(Counter(transition_status_name(t[-1]) for t in expert_trajectories))}")
verify_next_observation(expert_trajectories, label="Expert")

In [ ]:
EXPERT_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(EXPERT_SAVE_PATH, "wb") as f:
    pickle.dump(expert_trajectories, f)
print(f"Salvato: {EXPERT_SAVE_PATH}")
print(f"  {len(expert_trajectories)} traiettorie, {sum(len(t) for t in expert_trajectories):,} transizioni")

## 2. Balanced Eval Dataset

Raccolta in tre passate distinte:

| Passata | Policy | Target |
|---|---|---|
| A | Expert puro (`eps=0`) | `running`, `arrived` |
| B | Expert + esplorazione (`eps=0.85`) | `collision`, `offroad` |
| C | `bernoulli_50k seed=2` (checkpoint) | **`timeout`** |

L'agente della passata C è l'**unico** tra i 30 dell'esperimento a produrre timeout  
(50 timeout su 54 episodi in 30k step — `ppo_chri_binary_bernoulli q=50000 seed=2`).

In [14]:
# ── Passate A e B (expert + esplorazione) ──────────────────────────────────
buckets_by_length = {
    seg_len: {label: [] for label in TARGET_CLASSES}
    for seg_len in SEGMENT_LENGTHS
}


def extract_segments(traj, seg_len):
    return [Trajectory(traj[i: i + seg_len]) for i in range(len(traj) - seg_len + 1)]


def fill_buckets(trajectories):
    for traj in trajectories:
        for seg_len in SEGMENT_LENGTHS:
            if len(traj) < seg_len:
                continue
            for seg in extract_segments(traj, seg_len):
                label = segment_status_name(seg)
                if label in TARGET_CLASSES:
                    buckets_by_length[seg_len][label].append(seg)


def print_buckets():
    import pandas as pd
    rows = []
    for sl in SEGMENT_LENGTHS:
        row = {"seg_len": sl}
        row.update({c: len(buckets_by_length[sl][c]) for c in TARGET_CLASSES})
        rows.append(row)
    display(pd.DataFrame(rows).set_index("seg_len"))


EVAL_PASSES = [
    (0.0,  N_EVAL_STEPS_PER_PASS, "expert puro"),
    (0.85, N_EVAL_STEPS_PER_PASS, "esplorazione 0.85"),
]

t0_all = time.perf_counter()
for pass_idx, (eps, n_steps, label) in enumerate(EVAL_PASSES):
    print(f"\nPassata {pass_idx + 1}/{len(EVAL_PASSES)} — {label} ({n_steps:,} step)")
    t0 = time.perf_counter()
    if eps == 0.0:
        trajs = collect_trajectories(
            policy=expert_policy, n_steps=n_steps,
            n_envs=N_EVAL_ENVS, seed=EVAL_SEED + pass_idx, deterministic=True,
        )
    else:
        trajs = collect_trajectories_with_exploration(
            policy=expert_policy, n_steps=n_steps, n_envs=N_EVAL_ENVS,
            exploration_eps=eps, seed=EVAL_SEED + pass_idx,
        )
    fill_buckets(trajs)
    term = Counter(transition_status_name(t[-1]) for t in trajs)
    print(f"  {len(trajs)} traj in {time.perf_counter() - t0:.1f}s — terminali: {dict(term)}")
    verify_next_observation(trajs, label=f"  passata {pass_idx+1}")

print(f"\nBuckets dopo passate A+B:")
print_buckets()


Passata 1/2 — expert puro (50,000 step)

Environment closed.
Environment closed.
Environment closed.
Environment closed.



  308 traj in 18.8s — terminali: {'arrived': 272, 'collision': 36}
  passata 1: 308 traj, 50315 trans — next_observation: OK

Passata 2/2 — esplorazione 0.85 (50,000 step)

Environment closed.
Environment closed.
Environment closed.



Environment closed.
  2350 traj in 55.8s — terminali: {'collision': 1070, 'offroad': 1280}
  passata 2: 2350 traj, 50551 trans — next_observation: OK

Buckets dopo passate A+B:


,running,arrived,collision,offroad,timeout
seg_len,,,,,
1,98208,272,1106,1280,0
5,88649,272,905,1019,0
20,65345,272,368,529,0


In [15]:
# ── Passata C — agente timeout ──────────────────────────────────────────────
# ppo_chri_binary_bernoulli q=50000 seed=2: produce ~50 timeout / 30k step
# Con N_TIMEOUT_STEPS=200k → ~330 timeout episodi → >300 segmenti timeout per ogni seg_len

print("Caricamento timeout agent...")
env_tmp = make_env(seed=EVAL_SEED, n_envs=1)
timeout_policy = PPO.load(str(TIMEOUT_AGENT_PATH), env=env_tmp, device="cpu")
env_tmp.close()
print("Caricato.")

print(f"\nPassata C — timeout agent ({N_TIMEOUT_STEPS:,} step)")
t0 = time.perf_counter()
timeout_trajs = collect_trajectories(
    policy=timeout_policy, n_steps=N_TIMEOUT_STEPS,
    n_envs=N_EVAL_ENVS, seed=EVAL_SEED + 10, deterministic=True,
)
fill_buckets(timeout_trajs)

term = Counter(transition_status_name(t[-1]) for t in timeout_trajs)
print(f"  {len(timeout_trajs)} traj in {time.perf_counter() - t0:.1f}s — terminali: {dict(term)}")
verify_next_observation(timeout_trajs, label="  passata C")

print("\nBuckets dopo passata C:")
print_buckets()

Caricamento timeout agent...

Environment closed.
Caricato.

Passata C — timeout agent (200,000 step)

Environment closed.
Environment closed.
Environment closed.
Environment closed.



  388 traj in 55.9s — terminali: {'collision': 52, 'timeout': 336}
  passata C: 388 traj, 202003 trans — next_observation: OK

Buckets dopo passata C:


,running,arrived,collision,offroad,timeout
seg_len,,,,,
1,299823,272,1158,1280,336
5,288730,272,942,1019,336
20,260231,272,369,529,336


In [16]:
# Campionamento bilanciato
available_per_length = {
    sl: min(len(buckets_by_length[sl][c]) for c in TARGET_CLASSES)
    for sl in SEGMENT_LENGTHS
}
actual_n = {sl: min(N_PER_CLASS, avail) for sl, avail in available_per_length.items()}

print("Disponibilità minima per classe:", available_per_length)
print("N per classe salvato:           ", actual_n)


def sample_balanced(buckets, n, rng):
    segs, labels = [], []
    for label in TARGET_CLASSES:
        bucket = buckets[label]
        if len(bucket) < n:
            raise ValueError(f"{label}: richiesti {n}, disponibili {len(bucket)}")
        idx = rng.choice(len(bucket), size=n, replace=False)
        for i in idx:
            segs.append(bucket[int(i)])
            labels.append(label)
    order = rng.permutation(len(segs))
    return [segs[int(i)] for i in order], [labels[int(i)] for i in order]


segments_by_length, labels_by_length = {}, {}
for sl in SEGMENT_LENGTHS:
    n = actual_n[sl]
    if n == 0:
        print(f"ATTENZIONE: seg_len={sl} saltato — almeno una classe è vuota")
        continue
    segs, labs = sample_balanced(buckets_by_length[sl], n, rng)
    segments_by_length[sl] = segs
    labels_by_length[sl]   = labs
    print(f"seg_len={sl}: {len(segs)} segmenti — {Counter(labs)}")

# Verifica next_observation
all_trans = [tr for segs in segments_by_length.values() for s in segs for tr in s]
missing = sum(1 for tr in all_trans if tr.next_observation is None)
print(f"\nnext_observation mancante: {missing} / {len(all_trans)} — {'OK' if missing == 0 else 'ERRORE'}")

Disponibilità minima per classe: {1: 272, 5: 272, 20: 272}
N per classe salvato:            {1: 272, 5: 272, 20: 272}
seg_len=1: 1360 segmenti — Counter({'arrived': 272, 'running': 272, 'collision': 272, 'offroad': 272, 'timeout': 272})
seg_len=5: 1360 segmenti — Counter({'timeout': 272, 'running': 272, 'collision': 272, 'offroad': 272, 'arrived': 272})
seg_len=20: 1360 segmenti — Counter({'offroad': 272, 'running': 272, 'collision': 272, 'timeout': 272, 'arrived': 272})

next_observation mancante: 0 / 35360 — OK


In [17]:
balanced_eval_dataset = {
    "metadata": {
        "expert_model":         str(EXPERT_MODEL_PATH),
        "timeout_agent":        str(TIMEOUT_AGENT_PATH),
        "passes":               EVAL_PASSES,
        "n_timeout_steps":      N_TIMEOUT_STEPS,
        "n_envs":               N_EVAL_ENVS,
        "requested_n_per_class": N_PER_CLASS,
        "actual_n_per_class":   actual_n,
        "segment_lengths":      SEGMENT_LENGTHS,
        "target_classes":       TARGET_CLASSES,
        "rng_seed":             RNG_SEED,
        "next_observation":     True,
    },
    "segments_by_length": segments_by_length,
    "labels_by_length":   labels_by_length,
}

EVAL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(EVAL_SAVE_PATH, "wb") as f:
    pickle.dump(balanced_eval_dataset, f)

print(f"Salvato: {EVAL_SAVE_PATH}")
for sl, labs in labels_by_length.items():
    print(f"  seg_len={sl}: {Counter(labs)}")

Salvato: /Users/andreazhang/Desktop/newTesi/sumo-human-feedback-rl/data_for_training/balanced_eval_dataset_airl.pkl
  seg_len=1: Counter({'arrived': 272, 'running': 272, 'collision': 272, 'offroad': 272, 'timeout': 272})
  seg_len=5: Counter({'timeout': 272, 'running': 272, 'collision': 272, 'offroad': 272, 'arrived': 272})
  seg_len=20: Counter({'offroad': 272, 'running': 272, 'collision': 272, 'timeout': 272, 'arrived': 272})


## Verifica finale

In [18]:
print("=" * 60)
print("VERIFICA FINALE")
print("=" * 60)

with open(EXPERT_SAVE_PATH, "rb") as f:
    et = pickle.load(f)
et_trans = [tr for traj in et for tr in traj]
et_miss  = sum(1 for tr in et_trans if tr.next_observation is None)
print(f"\nexpert_trajectories_airl.pkl")
print(f"  traiettorie  : {len(et)}")
print(f"  transizioni  : {len(et_trans):,}")
print(f"  obs shape    : {et_trans[0].observation.shape}")
print(f"  next_obs     : {'OK' if et_miss == 0 else f'ERRORE ({et_miss} mancanti)'}")

with open(EVAL_SAVE_PATH, "rb") as f:
    ev = pickle.load(f)
print(f"\nbalanced_eval_dataset_airl.pkl")
for sl, segs in ev["segments_by_length"].items():
    miss_eval = sum(1 for s in segs for tr in s if tr.next_observation is None)
    print(f"  seg_len={sl}: {len(segs)} segmenti, next_obs={'OK' if miss_eval == 0 else f'ERRORE ({miss_eval})'}")
    print(f"    classi: {Counter(ev['labels_by_length'][sl])}")

print("\nDone.")

VERIFICA FINALE

expert_trajectories_airl.pkl
  traiettorie  : 3083
  transizioni  : 500,465
  obs shape    : (16,)
  next_obs     : OK

balanced_eval_dataset_airl.pkl
  seg_len=1: 1360 segmenti, next_obs=OK
    classi: Counter({'arrived': 272, 'running': 272, 'collision': 272, 'offroad': 272, 'timeout': 272})
  seg_len=5: 1360 segmenti, next_obs=OK
    classi: Counter({'timeout': 272, 'running': 272, 'collision': 272, 'offroad': 272, 'arrived': 272})
  seg_len=20: 1360 segmenti, next_obs=OK
    classi: Counter({'offroad': 272, 'running': 272, 'collision': 272, 'timeout': 272, 'arrived': 272})

Done.
